# IELTS Band Score Prediction
### Predicting student IELTS performance using ML — Linear Regression, Random Forest & XGBoost

---

**Goal:** Build a model that predicts a student's final IELTS band score (1–9) from assessment records and behavioural learning features.

**Dataset:** 5,000 synthetic student records with 9 features.

**Approach:**
1. Exploratory Data Analysis
2. Feature Engineering
3. Model Training + 5-Fold Cross Validation
4. Evaluation: RMSE, MAE, R²
5. Feature Importance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
print('Libraries loaded.')

## 1. Load Data

In [ ]:
import os

# Works on Kaggle and locally
kaggle_path = '/kaggle/input/ielts-performance-dataset/dataset.csv'
local_path  = '../data/raw/dataset.csv'
path = kaggle_path if os.path.exists(kaggle_path) else local_path

df = pd.read_csv(path)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Dtypes:')
print(df.dtypes)

## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['Final_IELTS_Band'], bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel('Final IELTS Band')
ax.set_ylabel('Count')
ax.set_title('Distribution of Final IELTS Band Score')
plt.tight_layout()
plt.show()
print(f"Mean: {df['Final_IELTS_Band'].mean():.2f}  |  Std: {df['Final_IELTS_Band'].std():.2f}")

In [ ]:
# Feature distributions
features = [c for c in df.columns if c != 'Final_IELTS_Band']
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.flatten(), features):
    ax.hist(df[col], bins=30, color='teal', edgecolor='white')
    ax.set_title(col, fontsize=10)
plt.suptitle('Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(11, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Top predictors vs target
top = ['Reading_Score', 'Listening_Score', 'Mock_Test_Score', 'Practice_Hours']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feat in zip(axes, top):
    ax.scatter(df[feat], df['Final_IELTS_Band'], alpha=0.25, s=8, color='steelblue')
    m, b = np.polyfit(df[feat], df['Final_IELTS_Band'], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, m*x_line + b, color='crimson', linewidth=1.5)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Final IELTS Band')
    ax.set_title(feat, fontsize=9)
plt.suptitle('Top Features vs Final IELTS Band', fontsize=12)
plt.tight_layout()
plt.show()

**Key EDA observations:**
- Final IELTS Band is roughly normally distributed with mean ~5.2
- Reading, Listening and Mock Test Score have the strongest linear correlation with the target
- Attendance, Practice Hours and Grammar also show meaningful positive relationships
- No missing values — dataset is clean

## 3. Feature Engineering

In [ ]:
df_fe = df.copy()

# Average of the four IELTS skill scores
df_fe['Avg_Skill_Score'] = df_fe[['Reading_Score','Writing_Score',
                                   'Listening_Score','Speaking_Score']].mean(axis=1)

# Combined language ability proxy
df_fe['Language_Ability'] = (df_fe['Vocabulary_Score'] + df_fe['Grammar_Score']) / 2

# Engagement index: attendance × practice hours
df_fe['Engagement_Index'] = (df_fe['Attendance_Percentage'] / 100) * df_fe['Practice_Hours']

print('New features added:', ['Avg_Skill_Score', 'Language_Ability', 'Engagement_Index'])
df_fe[['Avg_Skill_Score','Language_Ability','Engagement_Index','Final_IELTS_Band']].head()

## 4. Preprocessing

In [ ]:
X = df_fe.drop(columns=['Final_IELTS_Band'])
y = df_fe['Final_IELTS_Band']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 5. Model Training & 5-Fold Cross Validation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=SEED),
    'XGBoost':           XGBRegressor(n_estimators=200, learning_rate=0.05,
                                      max_depth=4, random_state=SEED, verbosity=0),
}

results = {}

for name, model in models.items():
    # Linear Regression uses scaled features; tree models don't need scaling
    X_tr = X_train_sc if name == 'Linear Regression' else X_train
    X_te = X_test_sc  if name == 'Linear Regression' else X_test
    X_cv = X_train_sc if name == 'Linear Regression' else X_train

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    # Test set metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    # 5-fold CV
    cv_rmse = np.sqrt(-cross_val_score(model, X_cv, y_train,
                                        cv=5, scoring='neg_mean_squared_error'))
    cv_r2   = cross_val_score(model, X_cv, y_train, cv=5, scoring='r2')

    results[name] = {
        'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'R2': round(r2, 4),
        'CV RMSE': f"{cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}",
        'CV R2':   f"{cv_r2.mean():.4f} ± {cv_r2.std():.4f}",
    }
    print(f"{name}: RMSE {rmse:.4f}  MAE {mae:.4f}  R2 {r2:.4f}  "
          f"CV RMSE {cv_rmse.mean():.4f}")

## 6. Results Comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
metric_colors = [('RMSE','#e74c3c'), ('MAE','#e67e22'), ('R2','#2ecc71')]

for ax, (metric, color) in zip(axes, metric_colors):
    vals  = [results[m][metric] for m in models]
    names = list(models.keys())
    bars  = ax.bar(names, vals, color=color, edgecolor='white')
    ax.set_title(metric, fontsize=12)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                str(val), ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Model Comparison — Test Set', fontsize=13)
plt.tight_layout()
plt.show()

## 7. XGBoost Feature Importance

In [ ]:
xgb_model = models['XGBoost']
feat_df = pd.DataFrame({
    'Feature':    list(X_train.columns),
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_df['Feature'], feat_df['Importance'],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

## 8. Summary

| Model | RMSE | MAE | R² | CV RMSE |
|---|---|---|---|---|
| **Linear Regression** | **0.158** | **0.128** | **0.981** | **0.153** |
| XGBoost | 0.164 | 0.134 | 0.980 | 0.163 |
| Random Forest | 0.169 | 0.137 | 0.979 | 0.169 |

**Key findings:**
- Linear Regression achieves the best performance — RMSE 0.158 and R² 0.981
- Reading Score, Listening Score and Mock Test Score are the strongest predictors of final band
- Attendance, Practice Hours and Grammar contribute meaningfully, confirming that behaviour matters alongside raw ability
- 5-fold cross-validation shows low variance across folds, confirming the model generalises well
- Composite features (Avg Skill Score, Language Ability, Engagement Index) add useful signal
